<a href="https://colab.research.google.com/github/zysdude/Scanner-Practice/blob/main/AI_Sanat_Ko%C3%A7u_%C3%87ocuklar_%C4%B0%C3%A7in_Yapay_Zeka_Destekli_Resim_Tamamlama_ve_Hikaye_Anlat%C4%B1m%C4%B1_Sistemi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
AI Sanat Koçu ve Tamamlayıcısı (Çocuklar İçin Fantastik Versiyon)
================================
Gemma 3 (4B Vision) + Stable Diffusion Img2Img pipeline
T4 GPU (16 GB VRAM) için optimize edilmiştir.

Bağımlılıklar:
    pip install torch torchvision transformers diffusers accelerate
    pip install Pillow sentencepiece bitsandbytes opencv-python numpy
"""

import gc
import os
import textwrap
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

In [ ]:
# ──────────────────────────────────────────────
# 0. YAPILANDIRMA
# ──────────────────────────────────────────────
INPUT_IMAGE   = "input_image.jpeg"
OUTPUT_IMAGE  = "final_art.jpg"

# T4 için 4-bit kuantizasyon ile Gemma 3 4B Vision
GEMMA_MODEL   = "google/gemma-3-4b-it"

# Hafif ama kaliteli bir SD modeli — T4'te img2img için uygun
SD_MODEL      = "runwayml/stable-diffusion-v1-5"

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE         = torch.float16 if DEVICE == "cuda" else torch.float32

# Çocuklar için güncellenmiş analiz ipuçları ve fantastik hikaye promptu
ART_COACH_PROMPT = (
    "Çocuklar için bir sanat koçu gibi davran. Bu karakalem eskizini analiz et. "
    "Çizimi daha da güzelleştirebilmeleri için çocuklara uygun, basit ve eğlenceli ipuçları ver "
    "(örneğin, renklendirme, detay ekleme). Ardından, bu manzarayı daha ilgi çekici ve "
    "çocukların hayal güçlerini ateşleyecek fantastik bir hikayeye dönüştür. Hikaye, "
    "resimdeki öğeleri (gökkuşağı, kayık, evler, ağaçlar) içermeli ve onların maceralarını anlatmalıdır."
)

In [ ]:
# ──────────────────────────────────────────────
# 1. GEMma VISION ANALİZİ (İpuçları + Hikaye)
# ──────────────────────────────────────────────

def analyze_with_gemma(image_path: str) -> str:
    """
    Gemma 3 vision modeliyle eskizi analiz eder ve
    çocuklara uygun ipuçları + fantastik bir macera hikayesi üretir.
    Belleği korumak için 4-bit kuantizasyon kullanılır.
    """
    from transformers import (
        AutoProcessor,
        AutoModelForImageTextToText,
        BitsAndBytesConfig,
    )

    print("[1/3] Gemma yükleniyor (4-bit kuantizasyon)...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    processor = AutoProcessor.from_pretrained(GEMMA_MODEL)
    model = AutoModelForImageTextToText.from_pretrained(
        GEMMA_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()

    image = Image.open(image_path).convert("RGB")

    # Gemma 3 chat formatı
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": ART_COACH_PROMPT},
            ],
        }
    ]

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=text_input,
        images=image,
        return_tensors="pt",
    ).to(DEVICE)

    print("   → Gemma analiz ve fantastik hikaye üretiyor...")
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=800, # Daha uzun hikaye için artırıldı
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
        )

    # Yalnızca yeni üretilen token'ları decode et
    generated = output_ids[:, inputs["input_ids"].shape[-1]:]
    description = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

    print("   ✓ Gemma işlemi tamamlandı.\n")
    print("─" * 60)
    print(description)
    print("─" * 60 + "\n")

    # Belleği serbest bırak
    del model, processor, inputs, output_ids
    gc.collect()
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

    return description

In [ ]:
# ──────────────────────────────────────────────
# 2. STABLE DIFFUSION IMG2IMG (Fantastik)
# ──────────────────────────────────────────────

def build_sd_prompt(gemma_text: str) -> str:
    """
    Stable Diffusion 1.5 sadece İngilizce anladığı ve 77 token sınırı olduğu için,
    Gemma'nın uzun Türkçe metnini kullanmak yerine doğrudan fantastik
    ve çocuk kitabına uygun güçlü bir İngilizce prompt veriyoruz.
    """
    return (
        "magical fantasy landscape, children's book illustration, whimsical style, "
        "vibrant colors, enchanted, beautiful lighting, highly detailed, 4k, masterpiece"
    )
    return f"{core}, {style_suffix}"


def generate_image_with_sd(prompt: str, input_path: str, output_path: str) -> Image.Image:
    """
    Stable Diffusion 1.5 Img2Img ile görsel üretir.
    Orijinal eskizi (input_path) baz alır.
    Attention slicing + sequential offload ile T4 belleği korunur.
    """
    # txt2img yerine img2img pipeline'ı kullanılıyor
    from diffusers import StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler

    print("[2/3] Stable Diffusion (Img2Img) yükleniyor...")

    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        SD_MODEL,
        torch_dtype=DTYPE,
        safety_checker=None,          # VRAM tasarrufu
        requires_safety_checker=False,
    )

    # T4 optimizasyonları
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_attention_slicing(1)   # Her adımda tek dilim → bellek düşer
    if DEVICE == "cuda":
        pipe.enable_sequential_cpu_offload()   # Katmanları CPU↔GPU arasında taşı

    print("   → Görsel (img2img) üretiliyor...")

    # Giriş görüntüsünü yükle ve 512x512'ye yeniden boyutlandır (SD 1.5 için)
    init_image = Image.open(input_path).convert("RGB").resize((512, 512))

    negative_prompt = (
        "grayscale, sketchy, pencil lines, watermark, text, low quality, ugly, blurry"
    )

    result = pipe(
        prompt=prompt,
        image=init_image, # init_image eklenmeli
        strength=0.5, # strength=0.5: Orijinali korumak ve fantastik eklemek için dengeli
        negative_prompt=negative_prompt,
        num_inference_steps=30,       # Kalite/hız dengesi
        guidance_scale=7.5,
        generator=torch.Generator(device=DEVICE).manual_seed(42),
    )

    art_image = result.images[0]
    art_image.save(output_path)
    print(f"   ✓ Görsel kaydedildi → {output_path}\n")

    del pipe
    gc.collect()
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

    return art_image

In [ ]:
# ──────────────────────────────────────────────
# 3. HİKAYE VE İPUÇLARI EKLEME (görsel altına metin)
# ──────────────────────────────────────────────

def add_story_to_image(image_path: str, story: str, final_path: str) -> None:
    """
    Üretilen görselin altına ipuçlarını ve fantastik hikayeyi ekler.
    Otomatik metin sarma ve uyumlu arka plan kullanır.
    """
    print("[3/3] Hikaye ve ipuçları görsele ekleniyor...")

    art = Image.open(image_path).convert("RGB")
    W, H = art.size

    # Metin alanı için parametreler
    MARGIN        = 20
    FONT_SIZE     = 16
    LINE_SPACING  = 6
    BG_COLOR      = (245, 235, 210)   # Krem / tuval tonu
    TEXT_COLOR    = (50, 30, 10)      # Koyu kahve

    try:
        # T4 / Ubuntu'da varsayılan font yolu
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf", FONT_SIZE)
    except OSError:
        print("   ! Font bulunamadı, varsayılan font kullanılıyor.")
        font = ImageFont.load_default()

    # Metni satırlara böl (İpuçları ve Hikaye dahil)
    wrapper  = textwrap.TextWrapper(width=int((W - 2 * MARGIN) / (FONT_SIZE * 0.55)))
    lines    = wrapper.wrap(story)

    line_h   = FONT_SIZE + LINE_SPACING
    text_h   = len(lines) * line_h + 2 * MARGIN + 20 # Başlık için ekstra alan

    # Yeni tuval
    canvas = Image.new("RGB", (W, H + text_h), BG_COLOR)
    canvas.paste(art, (0, 0))

    draw = ImageDraw.Draw(canvas)

    # İnce ayırıcı çizgi
    draw.line([(MARGIN, H + 8), (W - MARGIN, H + 8)], fill=(180, 140, 90), width=2)

    # Başlık (Güncellendi)
    title = "🎨  AI Sanat Koçu'ndan..."
    draw.text((MARGIN, H + MARGIN), title, font=font, fill=(100, 60, 20))

    y = H + MARGIN + line_h + 4
    for line in lines:
        draw.text((MARGIN, y), line, font=font, fill=TEXT_COLOR)
        y += line_h

    canvas.save(final_path, quality=95)
    print(f"   ✓ Final görsel kaydedildi → {final_path}\n")

In [ ]:
# ──────────────────────────────────────────────
# 4. HİKAYE EKLEME (FORMAT VE FONT DÜZELTMESİ)
# ──────────────────────────────────────────────
def add_story_to_image(img: Image.Image, story: str, final_path: str) -> None:
    """
    Madde işaretleri ve paragrafları bozmadan Türkçe karakter
    destekli font ile metni görselin altına ekler.
    """
    print("[4/4] Hikaye görsele ekleniyor...")

    W, H = img.size
    MARGIN        = 40
    FONT_SIZE     = 20
    LINE_SPACING  = 8
    BG_COLOR      = (245, 235, 210)
    TEXT_COLOR    = (50, 30, 10)

    # LiberationSans fontu - Türkçe karakter sorunu için
    font_path = "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"
    try:
        font = ImageFont.truetype(font_path, FONT_SIZE)
    except OSError:
        print("   ! Font bulunamadı, varsayılan fonta geçiliyor (Türkçe karakterler bozulabilir).")
        font = ImageFont.load_default()

    # Paragraf farkındalıklı Text Wrap
    max_chars = int((W - 2 * MARGIN) / (FONT_SIZE * 0.55))
    wrapper = textwrap.TextWrapper(width=max_chars)

    wrapped_lines = []
    for paragraph in story.split('\n'):
        if paragraph.strip() == "":
            wrapped_lines.append("")  # Boş satır (paragraf boşluğu)
        else:
            wrapped_lines.extend(wrapper.wrap(paragraph))

    line_h = FONT_SIZE + LINE_SPACING
    text_h = len(wrapped_lines) * line_h + (3 * MARGIN)

    # Yeni Tuval
    canvas = Image.new("RGB", (W, H + text_h), BG_COLOR)
    canvas.paste(img, (0, 0))

    draw = ImageDraw.Draw(canvas)
    draw.line([(MARGIN, H + 15), (W - MARGIN, H + 15)], fill=(180, 140, 90), width=3)

    # Başlık
    title = f"🎨 AI Sanat Koçu'nun Gözünden... (Mod: {MODE})"
    draw.text((MARGIN, H + MARGIN), title, font=font, fill=(100, 60, 20))

    # Metni Yazdırma
    y = H + MARGIN + line_h + 15
    for line in wrapped_lines:
        if line:
            draw.text((MARGIN, y), line, font=font, fill=TEXT_COLOR)
        y += line_h

    canvas.save(final_path, quality=98)
    print(f"   ✓ Final görsel kaydedildi → {final_path}\n")

In [ ]:
rm -rf ~/.cache/huggingface/hub/models--google--gemma-2-2b-it

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00


In [ ]:
# ──────────────────────────────────────────────
# ANA AKIŞ
# ──────────────────────────────────────────────

def main():
    # Başlangıçta MODE değişkenini tanımlayalım (add_story_to_image içinde kullanılıyor)
    global MODE
    MODE = "Fantastik Macera"

    # Dosya kontrolü
    if not Path(INPUT_IMAGE).exists():
        print(f"[HATA] '{INPUT_IMAGE}' dosyası bulunamadı!")
        print("Lütfen sol taraftaki dosya simgesine tıklayarak görselinizi yükleyin")
        print(f"ve isminin tam olarak '{INPUT_IMAGE}' olduğundan emin olun.")
        return

    print("=" * 60)
    print("  AI SANAT KOÇU VE TAMAMLAYICISI (FANTASY IMG2IMG)")
    print(f"  Cihaz: {DEVICE.upper()}  |  dtype: {DTYPE}")
    print("=" * 60 + "\n")

    # Adım 1 — Gemma analizi (İpuçları + Fantastik Hikaye)
    try:
        gemma_output = analyze_with_gemma(INPUT_IMAGE)
    except Exception as e:
        print(f"\n[HATA] Gemma analizi sırasında bir sorun oluştu: {e}")
        return

    # Adım 2 — SD görsel üretimi (Img2Img)
    sd_prompt    = build_sd_prompt(gemma_output)
    print(f"[SD Prompt]\n{sd_prompt}\n")
    temp_art_path = "temp_art.jpg"

    try:
        generate_image_with_sd(sd_prompt, INPUT_IMAGE, temp_art_path)
    except Exception as e:
        print(f"\n[HATA] Görsel üretimi (SD) sırasında bir sorun oluştu: {e}")
        return

    # Adım 3 — Hikaye ve ipuçları ekleme
    if os.path.exists(temp_art_path):
        try:
            generated_img = Image.open(temp_art_path).convert("RGB")
            add_story_to_image(generated_img, gemma_output, OUTPUT_IMAGE)
            os.remove(temp_art_path)
            print("=" * 60)
            print(f"  Pipeline tamamlandı!  →  {OUTPUT_IMAGE}")
            print("=" * 60)
        except Exception as e:
            print(f"\n[HATA] Hikaye ekleme sırasında bir sorun oluştu: {e}")
    else:
        print("[HATA] Üretilen geçici görsel dosyası bulunamadı.")


if __name__ == "__main__":
    main()

  AI SANAT KOÇU VE TAMAMLAYICISI (FANTASY IMG2IMG)
  Cihaz: CUDA  |  dtype: torch.float16

[1/3] Gemma yükleniyor (4-bit kuantizasyon)...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   → Görsel (img2img) üretiliyor...


  0%|          | 0/15 [00:00<?, ?it/s]

   ✓ Görsel kaydedildi → temp_art.jpg

[4/4] Hikaye görsele ekleniyor...
   ✓ Final görsel kaydedildi → final_art.jpg

  Pipeline tamamlandı!  →  final_art.jpg
